# Corpus check: can BigCodeBench carry this study?

**CPU only. Set Accelerator to `None`** so this costs nothing from your GPU quota.
**Internet ON** for the dataset and the analysers. Runs in roughly 20-30 minutes.

## Why this exists

Correctness measurement is not optional. Without it, an agent that "fixes" a
finding by deleting the function looks like a success. So the corpus is only
usable if its own reference solutions pass their own tests on this machine, and
BigCodeBench reaches into a long tail of third-party libraries that Kaggle's image
may not carry.

The second thing it settles is the number that actually sets the budget: **what
share of tasks carry a static-analysis finding at all.** Transfer can only be
measured on code that had something to fix, so that share decides how many tasks
must be generated to reach the 150 analysable ones the power calculation asks for.

## What it decides

| measurement | what it changes |
|---|---|
| reference solutions that run | whether correctness is measurable, and the real corpus size |
| finding hit rate per rule set | which rules the agent is shown, and the screening cost |
| semgrep availability | whether the cross-engine held-out arm survives |
| seconds per task | whether CPU-side test execution can overlap generation |

It produces no research result. It converts four assumptions into four numbers.


## 1. Install

In [ ]:
# CPU-only: no vLLM, no torch, no GPU.
import subprocess
for pkg in ("datasets", "ruff", "bandit", "semgrep", "radon"):
    r = subprocess.run(f"pip install -q {pkg}", shell=True, text=True)
    print(f"{pkg:10s} {'ok' if r.returncode == 0 else 'FAILED'}")


## 2. Load the corpus and learn its schema

In [ ]:
# --- Load the corpus and learn its schema rather than assume it. ---
# The exact field names of BigCodeBench are not something to take on trust, so
# this prints them and every later cell builds against what is actually there.
import json
import os
import random
import subprocess
import sys
import tempfile
import textwrap
import time

CORPUS_ID = "bigcode/bigcodebench"
SAMPLE_N = 60          # tasks to actually execute; enough to estimate rates
EXEC_TIMEOUT = 25      # seconds per task
WORKERS = max(2, (os.cpu_count() or 4))
SEED = 0

try:
    from datasets import get_dataset_config_names, load_dataset
except ImportError:
    subprocess.run("pip install -q datasets", shell=True, check=True)
    from datasets import get_dataset_config_names, load_dataset

configs = []
try:
    configs = get_dataset_config_names(CORPUS_ID)
    print(f"configs: {configs}")
except Exception as exc:
    print(f"could not list configs: {type(exc).__name__}: {exc}")

ds = None
for attempt in ([configs[-1]] if configs else []) + [None, "default"]:
    try:
        ds = load_dataset(CORPUS_ID, attempt) if attempt else load_dataset(CORPUS_ID)
        print(f"loaded with config={attempt!r}")
        break
    except Exception as exc:
        print(f"  config={attempt!r} failed: {type(exc).__name__}: {str(exc)[:120]}")

if ds is None:
    raise SystemExit(
        "Could not load BigCodeBench. Run the fallback cell at the bottom, which "
        "uses EvalPlus instead."
    )

split = list(ds.keys())[0]
records = ds[split]
print(f"\nsplit {split!r}: {len(records)} tasks")

FIELDS = list(records[0].keys())
print(f"\nfields: {FIELDS}")
for f in FIELDS:
    val = records[0][f]
    preview = str(val).replace("\n", " ")[:90]
    print(f"  {f:22s} {type(val).__name__:6s} {preview}")

# Fixed shuffle: every later cell, and any future extension run, draws from the
# same order, so a partial run is a uniform random sample rather than a biased
# prefix.
order = list(range(len(records)))
random.Random(SEED).shuffle(order)
print(f"\nshuffled with seed {SEED}; first ids: {[records[i].get('task_id') for i in order[:5]]}")


## 3. Do the reference solutions run?

In [ ]:
# --- Can the tests actually run here? ---
# Correctness measurement is not optional: without it, an agent that "fixes"
# findings by deleting the function looks like a success. So the corpus is only
# usable if its own reference solutions pass their own tests on this machine.
#
# BigCodeBench draws on a long tail of third-party libraries, and Kaggle's image
# will not have all of them. The number that matters is not "does it work" but
# "what share of tasks run", because that share is the real corpus size.
from concurrent.futures import ThreadPoolExecutor


def assemble(rec, strategy):
    """Build a runnable file from a record. Strategies are tried in order because
    the field layout is discovered at runtime rather than assumed."""
    g = rec.get
    if strategy == "prompt+canonical+test":
        head = g("complete_prompt") or g("code_prompt") or g("prompt") or ""
        return f"{head}\n{g('canonical_solution') or ''}\n\n{g('test') or ''}"
    if strategy == "canonical+test":
        return f"{g('canonical_solution') or ''}\n\n{g('test') or ''}"
    if strategy == "solution+test":
        return f"{g('solution') or ''}\n\n{g('test') or ''}"
    raise ValueError(strategy)


def make_runnable(source):
    """Append a unittest entry point when the test file relies on one."""
    if "unittest" in source and "__main__" not in source:
        source += "\n\nif __name__ == '__main__':\n    unittest.main(verbosity=0)\n"
    return source


def execute(source, timeout=EXEC_TIMEOUT):
    """Run a candidate file in its own process. Returns (ok, detail)."""
    d = tempfile.mkdtemp()
    path = os.path.join(d, "candidate.py")
    with open(path, "w") as fh:
        fh.write(make_runnable(source))
    try:
        r = subprocess.run([sys.executable, path], capture_output=True, text=True,
                           timeout=timeout, cwd=d)
        if r.returncode == 0:
            return True, "pass"
        tail = (r.stderr or r.stdout).strip().splitlines()
        last = tail[-1][:150] if tail else "no output"
        kind = "import_error" if "ModuleNotFoundError" in (r.stderr or "") else "fail"
        return False, f"{kind}: {last}"
    except subprocess.TimeoutExpired:
        return False, "timeout"
    except Exception as exc:
        return False, f"harness_error: {type(exc).__name__}: {exc}"


# Pick the assembly strategy on a handful of tasks before spending the sample.
probe_ids = order[:6]
best_strategy, best_hits = None, -1
for strategy in ("prompt+canonical+test", "canonical+test", "solution+test"):
    try:
        hits = sum(execute(assemble(records[i], strategy))[0] for i in probe_ids)
    except Exception as exc:
        print(f"  {strategy:26s} unusable: {type(exc).__name__}: {exc}")
        continue
    print(f"  {strategy:26s} {hits}/{len(probe_ids)} reference solutions pass")
    if hits > best_hits:
        best_strategy, best_hits = strategy, hits

if best_hits <= 0:
    raise SystemExit(
        "No assembly strategy makes the reference solutions pass. The corpus "
        "cannot measure correctness here; run the fallback cell."
    )
print(f"\nusing strategy: {best_strategy}")

# Now measure the real runnable share on a proper sample.
sample_ids = order[:SAMPLE_N]
print(f"\nexecuting {len(sample_ids)} reference solutions on {WORKERS} workers...")

t0 = time.time()
def run_idx(i):
    rec = records[i]
    ok, detail = execute(assemble(rec, best_strategy))
    return {"idx": i, "task_id": rec.get("task_id"), "ok": ok, "detail": detail}

with ThreadPoolExecutor(max_workers=WORKERS) as pool:
    exec_results = list(pool.map(run_idx, sample_ids))
elapsed = time.time() - t0

runnable = [r for r in exec_results if r["ok"]]
by_kind = {}
for r in exec_results:
    if not r["ok"]:
        kind = r["detail"].split(":")[0]
        by_kind[kind] = by_kind.get(kind, 0) + 1

print(f"\n{len(runnable)}/{len(exec_results)} reference solutions pass "
      f"({len(runnable) / len(exec_results):.0%})")
print(f"wall clock {elapsed:.0f}s for {len(sample_ids)} tasks "
      f"= {elapsed / len(sample_ids):.1f}s per task at {WORKERS} workers")
for kind, n in sorted(by_kind.items(), key=lambda kv: -kv[1]):
    print(f"  {kind:16s} {n}")

for r in exec_results:
    if not r["ok"]:
        print(f"    e.g. {r['task_id']}: {r['detail'][:110]}")
        break


## 4. How many tasks have anything to fix?

In [ ]:
# --- The number that sets the budget: how many tasks have anything to fix? ---
# The study can only measure whether a fix transfers on code that has findings in
# the first place. That share is the screening hit rate, and it decides how many
# tasks must be generated to reach the target of 150 analysable ones.
#
# Rule selection matters as much as the corpus. Ruff's S rules mirror Bandit,
# which makes them the sharp shown/held-out pair, but security findings alone may
# be rare in benign code. Broader selections find more and overlap less cleanly.
# Rather than guess, measure the hit rate under each and let the data choose.
RULE_SETS = {
    "S": "security only, mirrors Bandit most closely",
    "S,B": "security plus bugbear",
    "S,B,C90,PERF": "security, bugbear, complexity, performance",
}


def ruff_findings(path, select):
    r = subprocess.run(
        f"ruff check --select {select} --output-format=json --isolated {path}",
        shell=True, capture_output=True, text=True)
    try:
        return json.loads(r.stdout or "[]")
    except json.JSONDecodeError:
        return []


def bandit_findings(path):
    r = subprocess.run(f"bandit -q -f json {path}", shell=True,
                       capture_output=True, text=True)
    try:
        return json.loads(r.stdout or "{}").get("results", [])
    except json.JSONDecodeError:
        return []


def semgrep_findings(path):
    r = subprocess.run(f"semgrep --config=p/python --quiet --json --timeout=20 {path}",
                       shell=True, capture_output=True, text=True)
    try:
        return json.loads(r.stdout or "{}").get("results", [])
    except json.JSONDecodeError:
        return None      # None means semgrep could not run, distinct from zero


# Analyse the *reference* solutions of the tasks that actually run. These are
# human-curated, so they are cleaner than model output will be: treat the rate
# below as a lower bound on the screening hit rate the study will see.
targets = [r["idx"] for r in exec_results if r["ok"]] or order[:SAMPLE_N]
print(f"analysing {len(targets)} reference solutions\n")

rows = []
d = tempfile.mkdtemp()
for n, i in enumerate(targets):
    rec = records[i]
    src = assemble(rec, best_strategy).split("\n\nclass Test")[0]   # drop the test body
    path = os.path.join(d, f"t{n}.py")
    with open(path, "w") as fh:
        fh.write(src)
    row = {"task_id": rec.get("task_id"), "loc": src.count("\n") + 1}
    for name, select in [(k, k) for k in RULE_SETS]:
        row[f"ruff[{name}]"] = len(ruff_findings(path, select))
    row["bandit"] = len(bandit_findings(path))
    sg = semgrep_findings(path) if n < 25 else None    # semgrep is slow; sample it
    row["semgrep"] = len(sg) if sg is not None else -1
    rows.append(row)

print(f"{'rule set':16s} {'tasks with >=1':>15s} {'hit rate':>9s} {'mean findings':>14s}")
hit_rates = {}
for name in RULE_SETS:
    key = f"ruff[{name}]"
    hits = sum(1 for r in rows if r[key] > 0)
    mean = sum(r[key] for r in rows) / len(rows)
    hit_rates[name] = hits / len(rows)
    print(f"{name:16s} {hits:>10d}/{len(rows):<4d} {hits / len(rows):>8.0%} {mean:>14.1f}")

b_hits = sum(1 for r in rows if r["bandit"] > 0)
print(f"{'bandit':16s} {b_hits:>10d}/{len(rows):<4d} {b_hits / len(rows):>8.0%} "
      f"{sum(r['bandit'] for r in rows) / len(rows):>14.1f}")

sg_rows = [r for r in rows if r["semgrep"] >= 0]
if sg_rows:
    s_hits = sum(1 for r in sg_rows if r["semgrep"] > 0)
    print(f"{'semgrep':16s} {s_hits:>10d}/{len(sg_rows):<4d} {s_hits / len(sg_rows):>8.0%} "
          f"{sum(r['semgrep'] for r in sg_rows) / len(sg_rows):>14.1f}")
else:
    print(f"{'semgrep':16s} could not run: the cross-engine held-out arm is unavailable")

print(f"\nmean file length: {sum(r['loc'] for r in rows) / len(rows):.0f} lines")
print("\nThese are curated reference solutions, so model output should trip the")
print("analysers at least this often. Treat these as lower bounds.")


## 5. Verdict and cost

In [ ]:
# --- Verdict: is this corpus usable, and what does the run cost? ---
TARGET_ANALYSABLE = 150     # from the power calculation, not a guess
TOKENS_PER_ROUND = 600      # conservative; smoke test saw 40-175 on short prompts
# Sum of 1/throughput across the four measured models, in GPU-seconds per token.
SECONDS_PER_TOKEN_ALL_MODELS = 1 / 789 + 1 / 271 + 1 / 546 + 1 / 487

runnable_rate = len(runnable) / len(exec_results) if exec_results else 0.0

print(f"reference solutions that run here : {runnable_rate:.0%}")
for name in RULE_SETS:
    print(f"tasks with >=1 finding [{name:12s}]: {hit_rates[name]:.0%}")

# Pick the rule set that gives a workable hit rate without drowning the agent in
# trivia. Too low and screening cost explodes; too high and the findings are
# mostly formatting noise that says nothing about defects.
choice = None
for name in RULE_SETS:
    if 0.35 <= hit_rates[name] <= 0.95:
        choice = name
        break
if choice is None:
    choice = max(hit_rates, key=lambda k: hit_rates[k])

hit = hit_rates[choice]
print(f"\nselected rule set: {choice}  ({RULE_SETS[choice]})")

if runnable_rate < 0.5:
    print("\nSTOP. Fewer than half the reference solutions run here, so correctness")
    print("cannot be measured for most of the corpus. Run the fallback cell.")
elif hit < 0.15:
    print(f"\nSTOP. Only {hit:.0%} of tasks carry a finding even at the broadest rule")
    print("set, so screening would dominate the budget. Run the fallback cell, or")
    print("widen the rule selection before proceeding.")
else:
    # Screening: generate once per task, keep those with a finding AND a runnable
    # test. Repair rounds are spent only on survivors.
    usable_rate = max(hit * runnable_rate, 1e-6)
    to_screen = TARGET_ANALYSABLE / usable_rate
    rounds = to_screen * 1 + TARGET_ANALYSABLE * 4   # 1 screen + 2 ruff + 2 semgrep
    gpu_seconds = rounds * TOKENS_PER_ROUND * SECONDS_PER_TOKEN_ALL_MODELS
    overhead_min = 10 + 4 * 5                        # install plus four model loads

    print(f"\nTo reach {TARGET_ANALYSABLE} analysable tasks:")
    print(f"  usable share            {usable_rate:.0%}  (has a finding AND runs)")
    print(f"  tasks to screen         {to_screen:,.0f}")
    print(f"  generation rounds       {rounds:,.0f} per model set")
    print(f"  GPU time                {gpu_seconds / 3600:.1f} h")
    print(f"  plus overhead           {overhead_min} min")
    print(f"  TOTAL                   {gpu_seconds / 3600 + overhead_min / 60:.1f} h")

    if to_screen > len(records):
        print(f"\n  ! Needs {to_screen:,.0f} tasks but the corpus has {len(records)}.")
        print(f"    Lower the target or widen the rule set.")

    cpu_min = to_screen * (elapsed / len(exec_results)) / 60
    print(f"\n  CPU-side test execution adds roughly {cpu_min:.0f} min, overlappable "
          f"with generation.")

summary = {
    "corpus": CORPUS_ID, "n_tasks": len(records), "sample": len(exec_results),
    "runnable_rate": runnable_rate, "hit_rates": hit_rates,
    "selected_rule_set": choice, "assembly": best_strategy,
    "secs_per_task_exec": elapsed / len(exec_results) if exec_results else None,
    "semgrep_available": any(r["semgrep"] >= 0 for r in rows),
}
os.makedirs("/kaggle/working", exist_ok=True)
with open("/kaggle/working/corpus_check.json", "w") as fh:
    json.dump({"summary": summary, "rows": rows, "exec": exec_results}, fh, indent=2)
print("\nWrote /kaggle/working/corpus_check.json")


## 6. Fallback (run only if the verdict said STOP)

In [ ]:
# --- Fallback: EvalPlus. Run this ONLY if the verdict cell said STOP. ---
# EvalPlus (HumanEval+ / MBPP+) is trivial to run: pure-Python tasks, no exotic
# dependencies, so the runnable share should be near total.
#
# The cost is construct validity. Its tasks are short standalone functions, which
# give a static analyser very little to find, so the study would be measuring
# transfer on a handful of findings per file rather than on realistic code. That
# is a genuine weakening of the result and belongs in the limitations section, not
# hidden in a config.
for cid in ("evalplus/humanevalplus", "evalplus/mbppplus"):
    try:
        alt = load_dataset(cid)
        split = list(alt.keys())[0]
        recs = alt[split]
        print(f"{cid}: {len(recs)} tasks, fields {list(recs[0].keys())}")

        d = tempfile.mkdtemp()
        hits = 0
        n = min(40, len(recs))
        for i in range(n):
            body = (recs[i].get("canonical_solution") or recs[i].get("solution") or "")
            head = recs[i].get("prompt") or ""
            path = os.path.join(d, f"f{i}.py")
            with open(path, "w") as fh:
                fh.write(head + body)
            if len(ruff_findings(path, "S,B,C90,PERF")) > 0:
                hits += 1
        print(f"  {hits}/{n} carry a finding at the broad rule set ({hits / n:.0%})")
        print(f"  mean length {sum((recs[i].get('prompt') or '').count(chr(10)) for i in range(n)) / n:.0f} lines")
    except Exception as exc:
        print(f"{cid}: {type(exc).__name__}: {str(exc)[:140]}")

print("\nIf the hit rate here is also low, the problem is the rule selection rather")
print("than the corpus, and the fix is to widen it rather than to change dataset.")


## Reading the verdict

**Both rates healthy** -> the corpus is confirmed and the printed GPU estimate is
built from your own measured throughput rather than from an assumption.

**Runnable share below 50%** -> correctness cannot be measured for most of the
corpus. Run the fallback cell.

**Hit rate below 15% everywhere** -> the problem is usually the rule selection
rather than the dataset. Widening the rules is the first thing to try; changing
corpus is the second.

**Semgrep unavailable** -> the study still works on the Ruff/Bandit pair, which is
the sharper of the two comparisons anyway, but it loses the cross-engine arm and
the write-up has to say so.

## A caveat worth carrying into the paper

The hit rates here are measured on curated reference solutions. Model-generated
code is generally messier, so the real screening hit rate should be at least this
high. Every estimate downstream treats these as lower bounds.
